![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 1. Parte A: De texto a tensor

**Asignatura:** Procesamiento del Lenguaje Natural
**Bloque:** 1 — Introducción a las Redes Neuronales

---

## Introducción

Una red neuronal no sabe leer. Lo único que sabe hacer es multiplicar matrices de números en punto flotante. Entre una frase como *"apagá la luz de la cocina"* y la primera capa de un modelo hay una cadena de decisiones —cómo se corta el texto en unidades, qué unidades entran al vocabulario, qué se hace con las que quedan afuera, cómo se le da forma rectangular a un conjunto de frases de largos distintos— y cada una de esas decisiones se paga después, en la calidad del modelo.

Esa cadena es el tema de esta primera parte. No vas a entrenar nada todavía: vas a construir la maquinaria que convierte texto en el tensor `(B, L)` que la Parte B va a consumir, y a medir el costo de cada decisión que tomes en el camino.

Al completar este laboratorio vas a poder:

- Crear, transformar y consultar tensores de PyTorch, y anticipar cuándo dos tensores comparten memoria.
- Usar *broadcasting* para aplicar una máscara sobre un lote de vectores.
- Calcular gradientes con `autograd` y verificarlos contra la derivada analítica.
- Escribir tokenizadores y medir sus efectos sobre un corpus real en español.
- Implementar una clase `Vocabulario` completa, y elegir su umbral de frecuencia con datos y no a ojo.
- Truncar, rellenar y enmascarar un lote de textos, y cuantificar cuánto del tensor resultante es relleno.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para resolver cada ejercicio, consultá el material teórico de las Clases 1 y 2 de la Unidad 1.
- **Este laboratorio corre entero en CPU.** No hace falta GPU, y activarla no lo va a hacer más rápido.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas (enunciados, explicaciones, ejemplos provistos y el encabezado) **no se tocan**: la corrección se hace celda por celda de manera automática y modificar lo que no corresponde puede invalidar tu entrega.

Si necesitás probar algo fuera de una celda de actividad, hacelo en una copia aparte y revertí los cambios antes de entregar.

In [ ]:
# ─── Setup: imports ─────────────────────────────────────────────────────────
# Todo lo que usamos viene preinstalado en Colab. No hace falta instalar nada.
import re
import math
import collections
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

print(f"Versión de PyTorch: {torch.__version__}")
print(f"Versión de pandas:  {pd.__version__}")

In [ ]:
# ─── Setup: el corpus MASSIVE en español ────────────────────────────────────
# MASSIVE es un corpus de órdenes a un asistente de voz, recolectado por
# Amazon en 51 idiomas. Usamos la partición en español (es-ES) de la variante
# "scenario", donde cada orden está etiquetada con el dominio al que pertenece
# (alarm, iot, calendar, weather, ...).
#
# No es traducción automática: son órdenes escritas por hablantes nativos, con
# sus tipeos, sus acentos faltantes y su variabilidad real. Eso importa mucho
# para lo que vamos a medir en la Sección B.
#
# Los tres splits ya vienen separados por los autores del corpus. Pesa menos
# de 2 MB en total, así que la descarga es instantánea.
#
# Leemos primero de una copia en el repo de la materia y usamos el Hub como
# respaldo: es el mismo archivo, pero así la clase no se cae si alguno de los
# dos servidores no responde.
REPO = "https://github.com/javovelez/pln-labs/raw/main/datos"
HUB  = "https://huggingface.co/datasets/SetFit/amazon_massive_scenario_es-ES/resolve/main"


def leer_split(nombre):
    """Lee un split del corpus, del repo de la materia o del Hub como respaldo."""
    try:
        return pd.read_json(f"{REPO}/{nombre}.jsonl", lines=True)
    except Exception:
        return pd.read_json(f"{HUB}/{nombre}.jsonl", lines=True)


train = leer_split("train")
val   = leer_split("validation")
test  = leer_split("test")

# `label` es el índice numérico del escenario; `label_text`, su nombre.
# Esta lista traduce índice -> nombre y la vamos a reusar en la Parte B.
ESCENARIOS = (train[["label", "label_text"]]
              .drop_duplicates()
              .sort_values("label")["label_text"]
              .tolist())

print(f"entrenamiento: {len(train):>6,} órdenes")
print(f"validación:    {len(val):>6,} órdenes")
print(f"prueba:        {len(test):>6,} órdenes")
print(f"escenarios:    {len(ESCENARIOS)}")
print()
print(train.head(6).to_string(index=False))

largos = train.text.str.split().str.len()
print()
print(f"palabras por orden: media {largos.mean():.1f}, mediana {largos.median():.0f}, "
      f"percentil 95 {np.percentile(largos, 95):.0f}, máximo {largos.max()}")

---
## Sección A: Mecánica de tensores

Los primeros cuatro ejercicios son de mecánica pura, con tensores inventados. Están descontextualizados a propósito: el objetivo es que las operaciones te salgan sin pensarlas, porque a partir de la Sección B van a aparecer todas juntas y con texto adentro.

Prestá atención a los ejercicios 2 y 3 en particular. El 2 tiene una trampa que causa bugs muy difíciles de encontrar, y el 3 es exactamente la operación que hace el clasificador que vas a construir en la Parte B.

### Ejercicio 1 — Creación, forma y tipo de un tensor

**Objetivo:** Crear tensores de las tres maneras que vas a usar todo el curso y leer sus atributos de forma y tipo.

**Enunciado:**

1. **Creá un tensor a partir de datos que ya tenés.** Con `torch.tensor(...)`, construí `ids` a partir de la lista de listas `[[5, 12, 7, 0, 0], [9, 3, 0, 0, 0]]`, forzando el tipo `torch.long`. Este tensor imita un lote de dos frases ya codificadas y rellenadas con ceros.

2. **Creá un tensor de ceros.** Con `torch.zeros(...)`, construí `ceros` de forma `(4, 6)`, también de tipo `torch.long`.

3. **Creá un tensor aleatorio reproducible.** Fijá la semilla con `torch.manual_seed(0)` y creá `vectores = torch.rand(4, 6, 8)`. Imprimí su suma total redondeada a cuatro decimales; si fijaste la semilla correctamente, tiene que darte `92.1386`.

4. **Para cada uno de los tres tensores, imprimí** su forma (`.shape`), su tipo de dato (`.dtype`), su cantidad de dimensiones (`.ndim`) y su cantidad total de elementos (`.numel()`).

5. **Provocá el error de tipo a propósito.** Una capa de *embeddings* indexa una tabla, así que exige índices enteros. Creá `torch.nn.Embedding(10, 4)` y pasale `torch.rand(2, 3)` (que es `float32`) adentro de un `try/except`, e imprimí el tipo de excepción y su mensaje.

> **Pista:** El argumento `dtype` está disponible en todos los constructores (`torch.tensor`, `torch.zeros`, `torch.ones`). Para capturar el error del punto 5 alcanza con `except Exception as e:`; el nombre de la clase de la excepción sale de `type(e).__name__`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

`ids` y `vectores` tienen tipos distintos: `torch.long` uno, `torch.float32` el otro. ¿Por qué un tensor de índices de vocabulario **no puede** ser de punto flotante, mientras que un tensor de *embeddings* **tiene que** serlo?

*(Escribí tu respuesta acá)*

### Ejercicio 2 — Cambiar la forma, y la trampa de la memoria compartida

**Objetivo:** Manipular la forma de un tensor con `view`, `reshape`, `unsqueeze` y `squeeze`, y entender cuándo el resultado comparte memoria con el original.

**Enunciado:**

**Parte A — cambiar la forma.** Partí de `t = torch.arange(24).reshape(2, 3, 4)` e imprimí la forma resultante de cada una de estas operaciones:

1. `t.view(6, 4)`.
2. `t.reshape(-1)` — el `-1` significa "calculá vos esta dimensión".
3. `t.unsqueeze(-1)`, que agrega una dimensión de tamaño 1 al final.
4. Deshacer lo anterior con `.squeeze(-1)`, y verificar que volviste a la forma original.
5. Ahora transponé: `tt = t.transpose(1, 2)`. Intentá `tt.view(2, 12)` adentro de un `try/except` e imprimí el error. Después hacé `tt.reshape(2, 12)` y mostrá que sí funciona.

**Parte B — la trampa.** Ejecutá estos tres pasos e imprimí `a` después de cada modificación:

1. Creá `a = torch.arange(6)`.
2. Creá `b = a.view(2, 3)` y escribí `b[0, 0] = 99`. Imprimí `a`.
3. Creá `c = a.clone().view(2, 3)` y escribí `c[0, 1] = -1`. Imprimí `a`.

Cerrá comparando `a.data_ptr()` contra `b.data_ptr()` y contra `c.data_ptr()`. Ese método devuelve la dirección de memoria donde arrancan los datos del tensor.

> **Pista 1:** `view` exige que el tensor sea *contiguo* en memoria, es decir que sus elementos estén guardados uno detrás del otro en el orden en que se los recorre. `transpose` no mueve datos: solo cambia cómo se los recorre, y por eso rompe la contigüidad. `reshape` se las arregla igual, copiando si hace falta.

> **Pista 2:** En la Parte B no hace falta que expliques nada todavía; eso va en la pregunta de análisis. Limitate a mostrar los valores.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Imaginá que en un `Dataset` escribís algo así:

```python
def __getitem__(self, i):
    fila = self.X[i]           # una fila del tensor completo
    fila[fila == self.pad_id] = self.unk_id   # "arreglo" el relleno
    return fila, self.y[i]
```

Explicá qué le pasa a `self.X` a medida que el `DataLoader` recorre el corpus, y por qué el bug es especialmente difícil de encontrar. ¿Cómo lo arreglarías?

*(Escribí tu respuesta acá)*

### Ejercicio 3 — Broadcasting de una máscara sobre un lote de vectores

**Objetivo:** Aplicar una máscara `(B, L, 1)` sobre un lote de vectores `(B, L, E)` usando *broadcasting*, y calcular un promedio que ignore el relleno.

**Enunciado:**

Este ejercicio es la operación central del clasificador que vas a construir en la Parte B, aislada y con números chicos para que puedas verificarla a mano.

Partí de este lote de índices, donde el `0` representa relleno:

```python
lote = torch.tensor([[5, 12, 7, 0, 0],
                     [9,  3, 0, 0, 0]])
```

1. **Simulá los vectores.** Con `torch.manual_seed(0)`, creá `vecs = torch.randn(2, 5, 4)`. Pensalo como: 2 frases, 5 posiciones cada una, un vector de 4 dimensiones por posición.

2. **Construí la máscara con la forma correcta.** A partir de `lote`, armá `mascara` de forma `(2, 5, 1)` y tipo `float`, que valga 1 en las posiciones con token real y 0 en el relleno. Imprimí su forma.

3. **Aplicá la máscara** multiplicando `vecs * mascara`. Imprimí la forma del resultado y verificá que las filas de relleno quedaron efectivamente en cero.

4. **Calculá el promedio enmascarado**: sumá sobre la dimensión de la secuencia y dividí por la cantidad de tokens **reales** de cada frase. Protegé la división con `.clamp(min=1)`.

5. **Calculá el promedio ingenuo** `vecs.mean(dim=1)`, que divide por `L` sin mirar la máscara, e imprimí los dos resultados de la primera frase para compararlos.

6. **Verificá que tu promedio enmascarado es el correcto**: para la primera frase, calculá a mano el promedio de las 3 filas reales (`vecs[0, :3].mean(dim=0)`) y compará con `torch.allclose`.

> **Pista 1:** El *broadcasting* estira automáticamente las dimensiones de tamaño 1. Al multiplicar `(2, 5, 4)` por `(2, 5, 1)`, la última dimensión de la máscara se replica 4 veces: el mismo 0 o 1 se aplica a las 4 componentes del vector de esa posición. Por eso hace falta el `unsqueeze(-1)`: sin él, tenés `(2, 5)` contra `(2, 5, 4)` y las formas no alinean.

> **Pista 2:** La cantidad de tokens reales por frase es la suma de la máscara sobre la dimensión de la secuencia, y sale ya con la forma `(2, 1)` que hace falta para dividir.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

En este ejercicio los vectores de las posiciones de relleno eran ruido (`torch.randn` los llenó igual que a los demás). En el modelo real, `nn.Embedding(..., padding_idx=0)` garantiza que la fila del `<pad>` sea **todo ceros**.

Con esa garantía, ¿el promedio ingenuo `vecs.mean(dim=1)` deja de estar mal? Justificá, y explicá qué relación exacta habría entre el promedio ingenuo y el enmascarado en ese caso.

*(Escribí tu respuesta acá)*

### Ejercicio 4 — Autograd: verificar un gradiente y controlar su acumulación

**Objetivo:** Calcular gradientes con `autograd`, verificarlos contra la derivada analítica, y entender por qué hay que ponerlos en cero en cada paso.

**Enunciado:**

**Parte A — verificación analítica.** Considerá la función $y = w^2 x$.

1. Derivá $\partial y / \partial w$ a mano (es una línea; escribila como comentario en el código).
2. Creá `w = torch.tensor([2.0], requires_grad=True)` y `x = torch.tensor([3.0])`.
3. Calculá `y` y llamá a `y.backward()`.
4. Imprimí `w.grad` junto al valor analítico y verificá que coinciden.

**Parte B — la acumulación.** Sin volver a crear el tensor:

1. Creá `w2 = torch.tensor([2.0], requires_grad=True)`.
2. Adentro de un bucle de tres iteraciones, recalculá `y` y llamá a `.backward()`, imprimiendo `w2.grad` en cada vuelta. Observá el patrón.
3. Poné el gradiente en cero (`w2.grad = None`), repetí el cálculo una vez más e imprimí el resultado.

> **Pista:** `backward()` **suma** el gradiente nuevo al que ya estaba guardado en `.grad`, no lo reemplaza. Es una decisión deliberada de PyTorch, no un descuido — pensá para qué puede servir acumular.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Si en un loop de entrenamiento te olvidás de llamar a `zero_grad()`, ¿qué le pasa al tamaño de los pasos de actualización a medida que avanzan las épocas? ¿Por qué el efecto se parece al de una tasa de aprendizaje mal elegida?

b) La acumulación es deliberada. Describí una situación de entrenamiento en la que **querés** que los gradientes se sumen antes de actualizar.

*(Escribí tu respuesta acá)*

---
## Sección B: De texto a tensor

Acá empieza el trabajo con el corpus real. La cadena tiene cuatro eslabones y los vas a construir en orden: **tokenizar** (cortar el texto en unidades), **armar el vocabulario** (decidir qué unidades existen y qué se hace con las demás), **codificar** (reemplazar cada unidad por su índice) y **darle forma rectangular** (truncar, rellenar y enmascarar).

Cada eslabón tiene un parámetro que parece menor y no lo es. La idea de esta sección es que ninguno lo elijas a ojo: en cada caso vas a medir el efecto antes de decidir.

### Ejercicio 5 — Dos tokenizadores, y qué le hace cada uno al corpus

**Objetivo:** Escribir dos tokenizadores con criterios distintos, observar su comportamiento sobre casos difíciles y medir la diferencia sobre el corpus completo.

**Enunciado:**

1. **Escribí `tok_simple(texto)`**: pasa el texto a minúsculas y devuelve todas las secuencias alfanuméricas. Una línea con `re.findall`.

2. **Escribí `tok_sin_acentos(texto)`**: hace lo mismo, pero además elimina los acentos. La receta estándar es normalizar a la forma `NFKD` con `unicodedata.normalize`, que separa cada letra acentuada en letra base + marca diacrítica, y después descartar los caracteres para los que `unicodedata.combining(c)` es distinto de cero.

3. **Probá los dos sobre esta lista de casos difíciles** e imprimí el resultado de cada uno. Son **órdenes reales del corpus**, elegidas porque cada una rompe algo distinto:

```python
dificiles = [
    "cuáles son las noticias en t. v. e. noticias",   # una sigla, partida en letras
    "cual es ese álbum de música actual",             # 'cual' sin tilde
    "cuales son las ultimas noticias",                # dos palabras sin tilde
    "envía un correo electronico a raul",             # tipeos sin tilde
]
```

4. **Medí los dos sobre el corpus de entrenamiento completo.** Para cada tokenizador, contá con `collections.Counter` cuántos tokens totales y cuántos tokens **únicos** produce sobre `train.text`, e imprimí una línea por tokenizador.

5. **Mostrá qué se fusionó.** Encontrá los grupos de palabras del corpus que colapsan en la misma forma al sacarles los acentos (por ejemplo `qué` y `que`), contá cuántos grupos hay e imprimí los cinco más frecuentes con su cantidad de apariciones.

> **Pista 1:** `r"\w+"` matchea secuencias de caracteres alfanuméricos, y en Python 3 incluye letras acentuadas y la `ñ`.

> **Pista 2:** Para el punto 5, agrupá las palabras del vocabulario por su versión sin acentos usando un `collections.defaultdict(list)`, y quedate con los grupos que tengan más de un elemento.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Mirá los grupos que colapsan. Algunos son claramente un error de tipeo que conviene unificar (`electrónico` / `electronico`), pero otros son **palabras distintas del español** que quedan fusionadas.

a) Identificá al menos dos grupos del segundo tipo y explicá qué información se pierde al fusionarlos.

b) Para este corpus en particular —órdenes cortas a un asistente de voz, escritas por gente que no siempre pone los acentos— ¿qué tokenizador elegirías y por qué? Justificá con los números que obtuviste, no con una preferencia general.

*(Escribí tu respuesta acá)*

### Ejercicio 6 — La clase `Vocabulario`

**Objetivo:** Implementar la clase que mapea tokens a índices y viceversa, medir el efecto del umbral de frecuencia, y elegirlo con datos.

**Enunciado:**

Es el ejercicio grande de esta parte. La clase que escribas acá la vas a volver a usar en la Parte B y en el Laboratorio 2, así que vale la pena que quede prolija.

**Parte A — la clase.** Implementá `Vocabulario` con esta interfaz:

- `__init__(self, textos, tokenizador=tok_simple, freq_min=1, max_tokens=None)`. Cuenta todos los tokens de `textos` en `self.contador`, se queda con los que aparecen al menos `freq_min` veces (y con los `max_tokens` más frecuentes, si se especifica) y arma las dos estructuras del mapeo: `self.itos` (lista, índice → token) y `self.stoi` (diccionario, token → índice).
- Los dos tokens especiales, `<pad>` y `<unk>`, van **primero** en `itos`, en ese orden. Guardá sus índices en `self.pad_id` y `self.unk_id`.
- `__len__`, y `__getitem__(self, token)` que devuelve el índice del token y **nunca lanza excepción**: lo desconocido cae en `unk_id`.
- `codificar(self, texto)`: string → lista de índices.
- `decodificar(self, indices, ocultar_pad=True)`: lista de índices o tensor → lista de tokens.
- `__repr__` informativo.

Probala: construí un vocabulario con `freq_min=1` sobre `train.text`, imprimí los primeros 12 tokens de `itos`, el índice de `"alarma"` y el de una palabra que no exista en el corpus.

**Parte B — cuánto pesa el umbral.** Agregá dos métricas a la clase:

- `cobertura` (propiedad): qué proporción de las **apariciones** del corpus con el que se construyó quedan cubiertas por el vocabulario.
- `tasa_unk(self, textos)`: qué proporción de los tokens de un texto **nuevo** cae en `<unk>`.

Con eso, imprimí una tabla con una fila por cada `freq_min` en `[1, 2, 3, 5, 10]` y cuatro columnas: umbral, tamaño del vocabulario, cobertura, y tasa de `<unk>` sobre `test.text`. Agregá al final cuántos tokens del corpus aparecen **una sola vez** (los *hapax*).

**Parte C — la elección y el viaje de ida y vuelta.** Elegí un `freq_min` justificable a partir de la tabla, construí el `vocab` definitivo con él (lo vas a usar en el Ejercicio 7) e imprimí su tamaño, su cobertura y su tasa de `<unk>` en test. Después codificá y decodificá estas tres frases, mostrando los índices y el texto reconstruido:

```python
frases = ["pon una alarma para las siete",
          "quiero un vuelo a katmandú el jueves",
          test.text.iloc[7]]
```

> **Pista 1:** `collections.Counter` tiene `.most_common()`, que devuelve los pares `(token, frecuencia)` ordenados de mayor a menor. Construir `itos` a partir de ahí garantiza que los índices bajos sean las palabras frecuentes, lo cual es cómodo para inspeccionar.

> **Pista 2:** Que los especiales vayan primero no es cosmético: garantiza que `<pad>` tenga el índice 0 sin importar cómo esté configurado el resto, y del 0 dependen el `torch.zeros` del Ejercicio 7 y el `padding_idx` del modelo de la Parte B.

> **Pista 3:** Para que `decodificar` acepte tanto listas como tensores, `torch.is_tensor(indices)` y `.tolist()` resuelven el caso.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Justificá el `freq_min` que elegiste, usando los números de tu tabla. ¿Qué le pasa a la tasa de `<unk>` a medida que subís el umbral, y por qué eso no es necesariamente malo?

b) La palabra `katmandú` volvió como `<unk>`. Un compañero propone arreglarlo poniendo `freq_min=1` para que ninguna palabra quede afuera. Explicá por qué eso **no** resuelve el problema de fondo, y qué le pasaría al modelo con las palabras que ve por primera vez en producción.

*(Escribí tu respuesta acá)*

### Ejercicio 7 — Truncar, rellenar y construir la máscara

**Objetivo:** Convertir un corpus de textos de largo variable en un tensor rectangular `(B, L)`, construir su máscara y medir cuánto del tensor es relleno.

**Enunciado:**

Un tensor es rectangular y las frases no lo son. Este ejercicio es el que resuelve esa tensión y produce, por fin, el tensor que la Parte B va a consumir.

1. **Escribí `codificar_lote(vocab, textos, largo)`**, que devuelve un tensor `(B, largo)` de tipo `torch.long` donde cada fila es un texto codificado, truncado a `largo` y rellenado con `pad_id`. Agregala a la clase con `Vocabulario.codificar_lote = ...` para poder llamarla como método.

2. **Medí el compromiso.** Para `largo` en `[8, 16, 32]`, aplicá la función sobre `train.text` completo y para cada valor imprimí: la forma del tensor, la proporción de posiciones que son relleno, y qué porcentaje de las órdenes quedaron truncadas (es decir, cuya codificación era más larga que `largo`).

3. **Elegí `L`** justificando la elección con esos números, y construí `X_train` sobre `train.text` con ese valor.

4. **Construí la máscara** `(X_train != vocab.pad_id)` y mostrá, para las primeras 5 órdenes: el tensor de índices, la máscara como enteros, y la cantidad de tokens reales por fila.

> **Pista 1:** Arrancá de `torch.zeros(len(textos), largo, dtype=torch.long)`. Como `<pad>` es el índice 0, el relleno ya viene hecho y solo hay que escribir los tokens reales encima con `salida[i, :len(ids)] = ...`.

> **Pista 2:** Para el porcentaje de truncadas te sirve `train.text.map(lambda t: len(vocab.codificar(t)))`, que te da el largo real de cada orden en tokens.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Con el `L` que elegiste, más de la mitad de las posiciones del tensor son relleno. Eso significa que más de la mitad del cómputo del modelo se va a gastar procesando posiciones que no significan nada.

a) ¿Por qué, aun así, conviene armar el tensor de esta manera en vez de procesar cada orden por separado con su largo real?

b) Proponé una manera de reducir ese desperdicio **sin** bajar `L`, y explicá qué habría que cambiar en el `DataLoader` para lograrlo.

*(Escribí tu respuesta acá)*

### Ejercicio 8 — Por qué un índice no es una representación

**Objetivo:** Cerrar la parte entendiendo qué logramos con la cadena que construimos y, sobre todo, qué **no** logramos.

**Enunciado:**

Terminaste el trabajo: cualquier orden en español entra por un lado y sale como una fila de un tensor `(B, L)` de enteros, que es exactamente lo que la Parte B necesita. Pero mirá esta tabla, que sale del vocabulario que construiste, antes de festejar:

| palabra | índice | frecuencia |
|---|---|---|
| `canción` | 45 | 281 |
| `alarma` | 46 | 274 |
| `música` | 52 | 239 |
| `luz` | 81 | 134 |
| `lámpara` | 856 | 9 |
| `despertador` | 1021 | 7 |

Respondé, apoyándote en esos números:

1. `alarma` y `despertador` son casi sinónimos en este dominio y quedaron a 975 posiciones de distancia; `alarma` y `música` no tienen nada que ver y quedaron a 6. Explicá qué codifica realmente el índice, por qué la resta entre dos índices no quiere decir nada, y qué relación falsa asumiría un modelo que recibiera estos números directamente como entrada.

2. La alternativa clásica es el vector *one-hot*: largo `len(vocab)`, un 1 en la posición del token y ceros en el resto. Eso arregla el problema del punto 1, pero trae dos problemas nuevos. Nombralos —estimando, para nuestro vocabulario y con `L = 16`, cuánta memoria ocuparía un lote de 64 órdenes en *one-hot* de `float32` frente a lo que ocupa como índices— y decí qué propiedad tendría que tener una buena representación, que ni los índices ni el *one-hot* tienen.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] Los tensores que imprimo tienen las formas y los tipos que pide cada enunciado (`torch.long` para índices).
- [ ] La clase `Vocabulario` corre completa: `codificar`, `decodificar`, `cobertura`, `tasa_unk` y `codificar_lote`.
- [ ] La tabla del Ejercicio 6b tiene sus cinco filas y la elección de `freq_min` está justificada con esos números.
- [ ] Los valores numéricos que imprimo son razonables (no hay infinitos, ni `NaN`, ni proporciones fuera de `[0, 1]`).
- [ ] Respondí las ocho preguntas de análisis (Ej. 1 a 8).
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Construiste la cadena completa que va de texto en español a un tensor que una red puede consumir. En el camino practicaste:

- **Mecánica de tensores**: creación, forma, tipo, memoria compartida, *broadcasting* y `autograd`.
- **Tokenización**, y cómo medir el efecto de una decisión de diseño en vez de discutirla en abstracto.
- **La clase `Vocabulario`**, con sus tokens especiales y su umbral de frecuencia elegido con datos.
- **Truncado, relleno y máscara**, y el compromiso entre perder información y desperdiciar cómputo.

Guardá tu implementación de `Vocabulario`: la Parte B te la da ya escrita en la celda de setup, pero conviene que compares.

En la **Parte B** ese tensor entra por fin a un modelo. Vas a construir un clasificador de escenarios con una tabla de *embeddings* y un promedio enmascarado —el mismo *broadcasting* del Ejercicio 3—, entrenarlo, medirlo con una matriz de confusión y provocarle un sobreajuste a propósito para después curarlo. Y al final vas a encontrarte con el límite que motiva toda la Unidad 2: el modelo que construyas no va a poder distinguir *"apagá la luz de la cocina"* de *"la cocina apagá de luz la"*.